# Lab 2: Vectorless RAG — Advanced Scenarios


## Setup

In [ ]:
!pip install -q pageindex openai python-dotenv pymupdf

In [ ]:
import os
import json
import re
import fitz  # PyMuPDF
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv("../.env")

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

print("Keys loaded.")

In [ ]:
def call_llm(prompt, model="meta-llama/llama-4-scout-17b-16e-instruct"):
    """Call a language model via OpenRouter."""
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
    )
    msgs = [{"role": "user", "content": prompt}]
    resp = client.chat.completions.create(model=model, messages=msgs, temperature=0, max_tokens=1024)
    return resp.choices[0].message.content.strip()

In [ ]:
PDF_PATH = "data/synthetic_medicare_plus_policy_detailed.pdf"

def extract_page_text(pdf_path):
    """Extract text from each PDF page."""
    doc = fitz.open(pdf_path)
    texts = {}
    for i in range(len(doc)):
        texts[i+1] = doc.load_page(i).get_text()
    doc.close()
    return texts

page_texts = extract_page_text(PDF_PATH)
print(f"Extracted text from {len(page_texts)} pages.")

In [ ]:
from pageindex import PageIndexClient
from pageindex import utils
import time

CACHE_DIR = "cache"
os.makedirs(CACHE_DIR, exist_ok=True)

def get_cache_path(pdf_path):
    pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
    return os.path.join(CACHE_DIR, f"{pdf_name}_tree.json")

def load_cached_tree(pdf_path):
    cache_path = get_cache_path(pdf_path)
    if os.path.exists(cache_path):
        with open(cache_path, "r") as f:
            tree = json.load(f)
        print(f"Loaded tree from cache: {cache_path}")
        return tree
    return None

def save_tree_to_cache(pdf_path, tree):
    cache_path = get_cache_path(pdf_path)
    with open(cache_path, "w") as f:
        json.dump(tree, f, indent=2)
    print(f"Tree saved to cache: {cache_path}")

In [ ]:
pi = PageIndexClient(api_key=PAGEINDEX_API_KEY)

tree = load_cached_tree(PDF_PATH)

if tree is None:
    print("Cache miss — submitting to PageIndex...")
    result = pi.submit_document(PDF_PATH)
    doc_id = result["doc_id"]
    print(f"Submitted: {doc_id}")

    print("Waiting for PageIndex to process...")
    elapsed = 0
    while elapsed < 300:
        if pi.is_retrieval_ready(doc_id):
            break
        time.sleep(5)
        elapsed += 5
        print(f"  {elapsed}s...")
    else:
        raise TimeoutError(f"PageIndex did not finish within {elapsed}s.")

    print(f"Ready after {elapsed}s")
    tree = pi.get_tree(doc_id, node_summary=True)["result"]
    save_tree_to_cache(PDF_PATH, tree)

utils.print_tree(tree, exclude_fields=["text"])

---
## Helper Functions

In [ ]:
def parse_json(text):
    """Extract JSON from LLM response."""
    text = re.sub(r"```json\s*|\s*```", "", text.strip())
    s, e = text.find("{"), text.rfind("}")
    if s != -1 and e != -1:
        text = text[s:e+1]
    return json.loads(text)

def retrieve_nodes(query, tree):
    """Use LLM to find relevant nodes for a query."""
    tree_slim = utils.remove_fields(tree.copy(), fields=["text"])
    search_prompt = f"""
You are given a question and a document tree.
Each node has: node_id, title, summary.
Find all nodes likely to contain the answer.

Question: {query}

Document tree:
{json.dumps(tree_slim, indent=2)}

Reply in this JSON format ONLY:
{{
    "thinking": "<your reasoning>",
    "node_list": ["node_id_1", "node_id_2"]
}}
"""
    raw = call_llm(search_prompt)
    result = parse_json(raw)
    return result["thinking"], result["node_list"]

def get_context(node_list, tree, page_texts):
    """Extract text from pages covered by the given nodes."""
    node_map = utils.create_node_mapping(tree, include_page_ranges=True, max_page=len(page_texts))
    texts, seen = [], set()
    for nid in node_list:
        info = node_map[nid]
        for p in range(info["start_index"], info["end_index"] + 1):
            if p not in seen and p in page_texts:
                texts.append(f"--- Page {p} ---\n{page_texts[p]}")
                seen.add(p)
    return "\n\n".join(texts)

def answer_query(query, context):
    """Use LLM to answer a query given context."""
    prompt = f"""
Answer the question based on the provided text.

Context:
{context}

Question: {query}

Rules:
- Answer only from the context
- If the answer isn't there, say so
- Be concise
"""
    return call_llm(prompt)

print("Helpers ready.")

---
# Scenario 1: Multi-Hop Attribute Aggregation

**Problem:** Some questions require combining information from multiple sections. For example, "What is the total premium including GST?" requires finding:
1. The base premium amount
2. The GST rate
3. Calculating the total

**Solution:** The LLM must retrieve nodes from multiple sections and aggregate the information.

In [ ]:
# Multi-hop question: requires info from Premium section AND GST section
QUERY_MULTI_HOP = "What is the total annual premium including 18% GST for a sum insured of 5 lakhs?"

In [ ]:
# Step 1: Retrieve relevant nodes — LLM should find BOTH premium and GST sections
thinking, nodes = retrieve_nodes(QUERY_MULTI_HOP, tree)

print("Reasoning:", thinking, "\n")
print("Retrieved nodes:")
for nid in nodes:
    print(f"  - {nid}")

In [ ]:
# Step 2: Get context from all retrieved nodes
context = get_context(nodes, tree, page_texts)
print(f"Context: {len(context.splitlines())} lines")

In [ ]:
# Step 3: LLM aggregates information from multiple sources
answer = answer_query(QUERY_MULTI_HOP, context)
print("Answer:", answer)

### Why Multi-Hop is Hard

Traditional RAG retrieves chunks by similarity. A query about "total premium with GST" might only match the premium section OR the GST section — not both. Vectorless RAG with tree-based reasoning can identify that the answer requires **multiple sections**.

---
# Scenario 2: Structured Data Fidelity

**Problem:** Documents contain tables, forms, and structured data. Extracting this data accurately is critical — a wrong number can be costly.

**Solution:** The LLM reads the raw text (which preserves table structure) and extracts values with high fidelity.

In [ ]:
# Structured data question: extract specific values from a table
QUERY_STRUCTURED = "What are the room rent limits for Private Hospital and ICU according to the table?"

In [ ]:
# Step 1: Retrieve nodes containing the room rent table
thinking, nodes = retrieve_nodes(QUERY_STRUCTURED, tree)

print("Reasoning:", thinking, "\n")
print("Retrieved nodes:")
for nid in nodes:
    print(f"  - {nid}")

In [ ]:
# Step 2: Get the context — raw text preserves table structure
context = get_context(nodes, tree, page_texts)
print(f"Context: {len(context.splitlines())} lines")

In [ ]:
# Step 3: LLM extracts structured data accurately
answer = answer_query(QUERY_STRUCTURED, context)
print("Answer:", answer)

### Why Structured Data Fidelity Matters

Tables in PDFs are often poorly formatted when extracted. Vectorless RAG preserves the original text flow, allowing the LLM to understand table structure from context (column headers, row labels, etc.).

---
# Scenario 3: Combined — Multi-Hop + Structured Data

The hardest case: a question that requires **both** multi-hop reasoning AND accurate structured data extraction.

In [ ]:
# Combined: find premium table AND calculate total
QUERY_COMBINED = "Based on the premium table, what is the total premium for a 25-year-old with 5 lakh sum insured including all taxes?"

In [ ]:
thinking, nodes = retrieve_nodes(QUERY_COMBINED, tree)

print("Reasoning:", thinking, "\n")
print("Retrieved nodes:")
for nid in nodes:
    print(f"  - {nid}")

In [ ]:
context = get_context(nodes, tree, page_texts)
answer = answer_query(QUERY_COMBINED, context)
print("Answer:", answer)

---
## Try It Yourself

Change the `QUERY_*` variables and re-run the cells.

In [ ]:
# Example multi-hop queries to try:
examples = [
    # Multi-Hop
    "What is the waiting period for PED and how does it reduce over years?",
    "What are the sub-limits for AYUSH treatment and room rent?",
    
    # Structured Data
    "List all the day care procedures covered with their limits.",
    "What is the copayment percentage for different age groups?",
    
    # Combined
    "Calculate the total deductible for a claim of 5 lakhs including all sub-limits.",
]

for q in examples:
    print(f"Query: {q}")
    thinking, nodes = retrieve_nodes(q, tree)
    print(f"  Nodes: {len(nodes)} retrieved")
    print()